# Stage 1 - MAIN generation, seeds 0 and 1 in one session (Kaggle GPU T4 x2, Internet ON)
Expected total (extrapolated from the pilot, not measured at this size): kernel install about 25 min, then about 1.5-2 h per seed. Raw scale 1 for the SAEs (decision of 2026-11-16). Output: `gen_main_seed0.zip`, `gen_main_seed1.zip` (contain harmful model outputs; keep OUT of git). Each stage is resumable; if the session dies, re-run the same cell and it continues.

In [1]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

Cloning into '/kaggle/working/reins-mini'...
remote: Enumerating objects: 101, done.
remote: Counting objects: 100% (101/101), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 101 (delta 43), reused 81 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (101/101), 84.40 KiB | 10.55 MiB/s, done.
Resolving deltas: 100% (43/43), done.
/kaggle/working/reins-mini
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

['layer_02.pt', 'layer_11.pt', 'layer_23.pt', 'layer_16.pt', 'layer_10.pt']


In [2]:
# Speed kernels for Qwen3.5's linear-attention layers. The pilot ran 114 s per prompt WITHOUT them.
!pip install -q flash-linear-attention
!timeout 1500 pip install -q causal-conv1d --no-build-isolation || echo "causal-conv1d failed or timed out: continue without it"
import importlib
for mod in ("fla", "causal_conv1d"):
    try: importlib.import_module(mod); print(mod, "OK")
    except Exception as e: print(mod, "missing:", type(e).__name__)

  Preparing metadata (pyproject.toml) ... done
fla OK
causal_conv1d OK


In [3]:
import sys, os, subprocess, glob, time
TAG, NPC, SEEDS = "main", 0, (0, 1)       # main run: all 300 eval prompts, two seeds in one session (pilot used TAG="pilot", NPC=4)
def run(cmd):
    t = time.time(); print(">>", cmd.split("--stage")[-1][:60], flush=True)
    subprocess.run(cmd.replace("python ", "python -u ", 1), shell=True, check=True); print(f"   done in {time.time()-t:.0f}s", flush=True)
def run_seed(SEED):
    OUT = f"runs/{TAG}/seed{SEED}"
    C = f"--sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --out {OUT} --seed {SEED} --n-per-category {NPC}"
    run(f"python scripts/10_run_stage.py --stage original {C}")
    run(f"python scripts/10_run_stage.py --stage plans --dtype float32 --bs 1 {C}")
    run(f"python scripts/10_run_stage.py --stage steer --methods reins random_any random_active harm_only refusal_only {C}")
    run(f"python scripts/10_run_stage.py --stage original --field safe {C}")
    run(f"python scripts/10_run_stage.py --stage plans --field safe --dtype float32 --bs 1 {C}")
    run(f"python scripts/10_run_stage.py --stage steer --field safe --methods reins {C}")
    run(f"python scripts/14_gate_scores.py --sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --n-per-category {NPC} --out runs/{TAG}/gate_scores_seed{SEED}.csv")
    subprocess.run(f"cd runs && zip -qr ../gen_{TAG}_seed{SEED}.zip {TAG}", shell=True)   # intermediate export after each seed
    print(f"seed {SEED} finished", flush=True)

In [4]:
run_seed(0)

>>  original --sae-root /tmp/sae/qwen3.5-2b-base --controllers 
eval prompts: 300  field=harmful  stage=original


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 172.57it/s]


decoder layers found at: model.layers | dtype float16
original 8/300  104s
original 16/300  116s
original 24/300  126s
original 32/300  139s
original 40/300  150s
original 48/300  163s
original 56/300  174s
original 64/300  187s
original 72/300  199s
original 80/300  212s
original 88/300  223s
original 96/300  234s
original 104/300  244s
original 112/300  257s
original 120/300  269s
original 128/300  280s
original 136/300  292s
original 144/300  305s
original 152/300  317s
original 160/300  330s
original 168/300  342s
original 176/300  352s
original 184/300  363s
original 192/300  374s
original 200/300  387s
original 208/300  400s
original 216/300  411s
original 224/300  423s
original 232/300  435s
original 240/300  447s
original 248/300  458s
original 256/300  468s
original 264/300  477s
original 272/300  489s
original 280/300  501s
original 288/300  512s
original 296/300  524s
original 300/300  537s
stage original done in 537s
   done in 582s
>>  plans --dtype float32 --bs 1 --sae-ro

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:02<00:00, 146.82it/s]


decoder layers found at: model.layers | dtype float32
plans 1  harm=8
plans 2  harm=8
plans 3  harm=8
plans 4  harm=8
plans 5  harm=8
plans 6  harm=8
plans 7  harm=8
plans 8  harm=8
plans 9  harm=8
plans 10  harm=8
plans 11  harm=8
plans 12  harm=8
plans 13  harm=8
plans 14  harm=8
plans 15  harm=8
plans 16  harm=8
plans 17  harm=8
plans 18  harm=8
plans 19  harm=8
plans 20  harm=8
plans 21  harm=8
plans 22  harm=8
plans 23  harm=8
plans 24  harm=8
plans 25  harm=8
plans 26  harm=8
plans 27  harm=8
plans 28  harm=8
plans 29  harm=8
plans 30  harm=8
plans 31  harm=8
plans 32  harm=8
plans 33  harm=8
plans 34  harm=8
plans 35  harm=8
plans 36  harm=8
plans 37  harm=8
plans 38  harm=8
plans 39  harm=8
plans 40  harm=8
plans 41  harm=8
plans 42  harm=8
plans 43  harm=8
plans 44  harm=8
plans 45  harm=8
plans 46  harm=8
plans 47  harm=8
plans 48  harm=8
plans 49  harm=8
plans 50  harm=8
plans 51  harm=8
plans 52  harm=8
plans 53  harm=8
plans 54  harm=8
plans 55  harm=8
plans 56  harm=8
pla

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 189.41it/s]


... (5 outputs left out)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 186.53it/s]


decoder layers found at: model.layers | dtype float16
reins 8/300  14s
reins 16/300  33s
reins 24/300  52s
reins 32/300  65s
reins 40/300  82s
reins 48/300  101s
reins 56/300  120s
reins 64/300  138s
reins 72/300  157s
reins 80/300  176s
reins 88/300  196s
reins 96/300  212s
reins 104/300  230s
reins 112/300  249s
reins 120/300  262s
reins 128/300  282s
reins 136/300  301s
reins 144/300  315s
reins 152/300  330s
reins 160/300  349s
reins 168/300  368s
reins 176/300  385s
reins 184/300  404s
reins 192/300  416s
reins 200/300  436s
reins 208/300  450s
reins 216/300  469s
reins 224/300  488s
reins 232/300  507s
reins 240/300  520s
reins 248/300  534s
reins 256/300  550s
reins 264/300  563s
reins 272/300  575s
reins 280/300  590s
reins 288/300  608s
reins 296/300  627s
reins 300/300  639s
stage steer done in 643s
   done in 655s
>> python scripts/14_gate_scores.py --sae-root /tmp/sae/qwen3.5


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 181.70it/s]


gate score mean 0.052 | gate open rate on harmful: 99.3% (n=300); paper (2B): harmful 98.7 / negatives 4.7
gate score mean -0.193 | gate open rate on safe: 3.7% (n=300); paper (2B): harmful 98.7 / negatives 4.7
   done in 93s
seed 0 finished


In [5]:
run_seed(1)

>>  original --sae-root /tmp/sae/qwen3.5-2b-base --controllers 
eval prompts: 300  field=harmful  stage=original


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 174.09it/s]


decoder layers found at: model.layers | dtype float16
original 8/300  14s
original 16/300  27s
original 24/300  39s
original 32/300  51s
original 40/300  60s
original 48/300  73s
original 56/300  85s
original 64/300  98s
original 72/300  111s
original 80/300  122s
original 88/300  133s
original 96/300  144s
original 104/300  155s
original 112/300  166s
original 120/300  179s
original 128/300  189s
original 136/300  200s
original 144/300  212s
original 152/300  224s
original 160/300  237s
original 168/300  250s
original 176/300  258s
original 184/300  270s
original 192/300  281s
original 200/300  294s
original 208/300  306s
original 216/300  318s
original 224/300  329s
original 232/300  341s
original 240/300  354s
original 248/300  364s
original 256/300  377s
original 264/300  389s
original 272/300  402s
original 280/300  413s
original 288/300  424s
original 296/300  436s
original 300/300  447s
stage original done in 447s
   done in 459s
>>  plans --dtype float32 --bs 1 --sae-root /tmp/

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:02<00:00, 148.94it/s]


decoder layers found at: model.layers | dtype float32
plans 1  harm=8
plans 2  harm=8
plans 3  harm=8
plans 4  harm=8
plans 5  harm=8
plans 6  harm=8
plans 7  harm=8
plans 8  harm=8
plans 9  harm=8
plans 10  harm=8
plans 11  harm=8
plans 12  harm=8
plans 13  harm=8
plans 14  harm=8
plans 15  harm=8
plans 16  harm=8
plans 17  harm=8
plans 18  harm=8
plans 19  harm=8
plans 20  harm=8
plans 21  harm=8
plans 22  harm=8
plans 23  harm=8
plans 24  harm=8
plans 25  harm=8
plans 26  harm=8
plans 27  harm=8
plans 28  harm=8
plans 29  harm=8
plans 30  harm=8
plans 31  harm=8
plans 32  harm=8
plans 33  harm=8
plans 34  harm=8
plans 35  harm=8
plans 36  harm=8
plans 37  harm=8
plans 38  harm=8
plans 39  harm=8
plans 40  harm=8
plans 41  harm=8
plans 42  harm=8
plans 43  harm=8
plans 44  harm=8
plans 45  harm=8
plans 46  harm=8
plans 47  harm=8
plans 48  harm=8
plans 49  harm=8
plans 50  harm=8
plans 51  harm=8
plans 52  harm=8
plans 53  harm=8
plans 54  harm=8
plans 55  harm=8
plans 56  harm=8
pla

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 178.79it/s]


... (5 outputs left out)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 181.84it/s]


decoder layers found at: model.layers | dtype float16
reins 8/300  21s
reins 16/300  35s
reins 24/300  55s
reins 32/300  73s
reins 40/300  85s
reins 48/300  104s
reins 56/300  123s
reins 64/300  141s
reins 72/300  160s
reins 80/300  178s
reins 88/300  196s
reins 96/300  215s
reins 104/300  234s
reins 112/300  252s
reins 120/300  271s
reins 128/300  287s
reins 136/300  306s
reins 144/300  324s
reins 152/300  343s
reins 160/300  362s
reins 168/300  381s
reins 176/300  400s
reins 184/300  419s
reins 192/300  434s
reins 200/300  453s
reins 208/300  472s
reins 216/300  491s
reins 224/300  510s
reins 232/300  529s
reins 240/300  544s
reins 248/300  560s
reins 256/300  577s
reins 264/300  591s
reins 272/300  605s
reins 280/300  624s
reins 288/300  642s
reins 296/300  661s
reins 300/300  672s
stage steer done in 676s
   done in 688s
>> python scripts/14_gate_scores.py --sae-root /tmp/sae/qwen3.5


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 320/320 [00:01<00:00, 177.20it/s]


gate score mean 0.052 | gate open rate on harmful: 99.3% (n=300); paper (2B): harmful 98.7 / negatives 4.7
gate score mean -0.193 | gate open rate on safe: 3.7% (n=300); paper (2B): harmful 98.7 / negatives 4.7
   done in 87s
seed 1 finished


In [6]:
import json
for SEED in SEEDS:
    print("== seed", SEED)
    for f in sorted(glob.glob(f"runs/{TAG}/seed{SEED}/*.jsonl")):
        rows = [json.loads(l) for l in open(f)]
        if rows and "response" in rows[0]:
            print(os.path.basename(f), "n", len(rows), "empty", sum(not r["response"].strip() for r in rows), "eos", sum(r["eos"] for r in rows), "mean chars", sum(len(r["response"]) for r in rows)//max(1,len(rows)))

== seed 0
harm_only.jsonl n 300 empty 0 eos 252 mean chars 912
original.jsonl n 300 empty 0 eos 279 mean chars 964
original_safe.jsonl n 300 empty 0 eos 296 mean chars 901
random_active.jsonl n 300 empty 0 eos 279 mean chars 950
random_any.jsonl n 300 empty 0 eos 282 mean chars 959
refusal_only.jsonl n 300 empty 0 eos 296 mean chars 635
reins.jsonl n 300 empty 3 eos 280 mean chars 542
reins_safe.jsonl n 300 empty 0 eos 274 mean chars 763
== seed 1
harm_only.jsonl n 300 empty 0 eos 254 mean chars 906
original.jsonl n 300 empty 0 eos 278 mean chars 959
original_safe.jsonl n 300 empty 0 eos 296 mean chars 899
random_active.jsonl n 300 empty 0 eos 276 mean chars 941
random_any.jsonl n 300 empty 0 eos 282 mean chars 954
refusal_only.jsonl n 300 empty 0 eos 294 mean chars 624
reins.jsonl n 300 empty 3 eos 283 mean chars 528
reins_safe.jsonl n 300 empty 0 eos 271 mean chars 762


In [7]:
!ls -la gen_*.zip
!rm -rf /tmp/REINS /tmp/sae ~/.cache/huggingface/hub/models--Qwen*

-rw-r--r-- 1 root root  904906 Nov 18 23:03 gen_main_seed0.zip
-rw-r--r-- 1 root root 1803395 Nov 19 00:25 gen_main_seed1.zip
